# Explore the ParlTrack dumps
Raw look at every file in this folder: the first records as-is, then the fields (columns) each file has.
Files are streamed, never loaded whole.

In [1]:
import io, json
from collections import Counter
from pathlib import Path
import pandas as pd
import zstandard

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

FOLDER = Path(".")
FILES = ["ep_amendments", "ep_plenary_amendments", "ep_meps", "ep_dossiers"]

def records(name, n=None):
    """Yield parsed records from <name>.json or <name>.json.zst, one per line."""
    plain, packed = FOLDER / f"{name}.json", FOLDER / f"{name}.json.zst"
    fh = open(plain, encoding="utf-8") if plain.exists() else \
        io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(open(packed, "rb")), encoding="utf-8")
    with fh:
        count = 0
        for line in fh:
            line = line.strip()
            if not line or line == "]":
                continue
            yield json.loads(line[1:])  # strip leading '[' or ','
            count += 1
            if n and count >= n:
                return

def show_raw(name, n=2):
    print(f"===== {name}: first {n} records =====")
    for r in records(name, n):
        print(json.dumps(r, indent=1, ensure_ascii=False))
        print("-" * 80)

def fields(name, sample=2000):
    """Which top-level fields exist, how often, their type and an example value."""
    seen, types, example = Counter(), {}, {}
    total = 0
    for r in records(name, sample):
        total += 1
        for k, v in r.items():
            seen[k] += 1
            types.setdefault(k, type(v).__name__)
            example.setdefault(k, json.dumps(v, ensure_ascii=False)[:120])
    return pd.DataFrame([{"field": k, "present_%": round(100 * c / total, 1), "type": types[k], "example": example[k]}
                         for k, c in seen.most_common()])

## `ep_amendments`

In [2]:
show_raw("ep_amendments", n=2)

===== ep_amendments: first 2 records =====
{
 "src": "http://www.europarl.europa.eu/sides/getDoc.do?pubRef=-//EP//NONSGML+COMPARL+PE-529.899+01+DOC+PDF+V0//EN&language=EN",
 "peid": "PE529.899v01-00",
 "reference": "2014/2021(INI)",
 "date": "2014-03-05T00:00:00",
 "committee": [
  "AFET"
 ],
 "seq": 1,
 "id": "PE529.899-1",
 "orig_lang": "en",
 "new": [
  "- having regard to its resolution of 13",
  "December 20071 on Justice for the",
  "'Comfort Women' (sex slaves in Asia",
  "before and during World War II) as well",
  "as the statements by Japanese Chief",
  "Cabinet Secretary Yohei Kono in 1993",
  "and by the then Prime Minister Tomiichi",
  "Murayama in 1995, the resolutions of the",
  "Japanese parliament (the Diet) of 1995",
  "and 2005 expressing apologies for",
  "wartime victims, including victims of the",
  "'comfort women' system,",
  "_______________________",
  "1",
  "OJ C 323E, 18.12.2008, p.531"
 ],
 "authors": "Reinhard Bütikofer on behalf of the Verts/ALE Group",


In [3]:
fields("ep_amendments")

,field,present_%,type,example
0,src,100.0,str,"""http://www.europarl.europa.eu/sides/getDoc.do?pubRef=-//EP//NONSGML+COMPARL+PE-529.899+01+DOC+PDF+V0//EN&language=EN"""
1,peid,100.0,str,"""PE529.899v01-00"""
2,reference,100.0,str,"""2014/2021(INI)"""
3,date,100.0,str,"""2014-03-05T00:00:00"""
4,committee,100.0,list,"[""AFET""]"
5,seq,100.0,int,1
6,id,100.0,str,"""PE529.899-1"""
7,orig_lang,100.0,str,"""en"""
8,authors,100.0,str,"""Reinhard Bütikofer on behalf of the Verts/ALE Group"""
9,location,100.0,list,"[[""Motion for a resolution"", ""Citation 6 a (new)""]]"


## `ep_plenary_amendments`

In [4]:
show_raw("ep_plenary_amendments", n=2)

===== ep_plenary_amendments: first 2 records =====
{
 "seq": "1",
 "location": [
  "Recital 1"
 ],
 "old": [
  "(1) Claiming to be “green” and sustainable has become a competitiveness factor, with green products registering greater growth than standard products. If goods and services offered and purchased on the internal market are not as environmentally friendly as presented, this would mislead the consumers, hamper the green transition and prevent the reduction of negative environmental impacts. The potential of green markets is not fully realised. Different requirements imposed by national legislation or private initiatives regulating environmental claims create a burden for companies in cross-border trade, as they need to comply with different requirements in each Member State. This affects their capacity to operate in and take advantage of the internal market. At the same time, market participants have difficulties with identifying reliable environmental claims and making optimal 

In [5]:
fields("ep_plenary_amendments")

,field,present_%,type,example
0,seq,100.0,str,"""1"""
1,location,100.0,list,"[""Recital 1""]"
2,src_type,100.0,str,"""html"""
3,reference,100.0,str,"""2023/0085(COD)"""
4,adoc_src,100.0,str,"""https://www.europarl.europa.eu/doceo/document/A-9-2024-0056_EN.html"""
5,id,100.0,str,"""A9-0056/2024-1"""
6,meta,100.0,dict,"{""created"": ""2024-12-31T00:58:07+00:00""}"
7,changes,100.0,dict,{}
8,new,99.8,list,"[""(1) Claiming to be “green” and sustainable has become a competitiveness factor, with green products registering gr..."
9,old,66.8,list,"[""(1) Claiming to be “green” and sustainable has become a competitiveness factor, with green products registering gr..."


## `ep_meps`

In [6]:
show_raw("ep_meps", n=1)  # MEP/dossier records are long

===== ep_meps: first 1 records =====
{
 "UserID": 2307,
 "Name": {
  "full": "Hubert PIRKER",
  "sur": "Hubert",
  "family": "PIRKER",
  "aliases": [
   "Hubert PIRKER",
   "PIRKER",
   "PIRKER Hubert"
  ]
 },
 "Photo": "https://www.europarl.europa.eu/mepphoto/2307.jpg",
 "meta": {
  "url": "http://www.europarl.europa.eu/meps/en/2307/name/cv",
  "updated": "2019-07-12T18:57:20"
 },
 "active": false,
 "Birth": {
  "date": "1948-10-03T00:00:00",
  "place": "Gries"
 },
 "Groups": [
  {
   "role": "Member",
   "Organization": "Group of the European People's Party (Christian-Democratic Group)",
   "groupid": "PPE",
   "start": "1996-11-11T00:00:00",
   "end": "1999-07-19T00:00:00"
  },
  {
   "role": "Member",
   "Organization": "Group of the European People's Party (Christian Democrats) and European Democrats",
   "groupid": "PPE-DE",
   "start": "1999-07-20T00:00:00",
   "end": "2004-07-19T00:00:00"
  },
  {
   "role": "Member",
   "Organization": "Group of the European People's Party (Ch

In [7]:
fields("ep_meps")

,field,present_%,type,example
0,UserID,100.0,int,2307
1,Name,100.0,dict,"{""full"": ""Hubert PIRKER"", ""sur"": ""Hubert"", ""family"": ""PIRKER"", ""aliases"": [""Hubert PIRKER"", ""PIRKER"", ""PIRKER Hubert""]}"
2,Photo,100.0,str,"""https://www.europarl.europa.eu/mepphoto/2307.jpg"""
3,meta,100.0,dict,"{""url"": ""http://www.europarl.europa.eu/meps/en/2307/name/cv"", ""updated"": ""2019-07-12T18:57:20""}"
4,active,100.0,bool,false
5,Constituencies,100.0,list,"[{""party"": ""Österreichische Volkspartei"", ""country"": ""Austria"", ""start"": ""1996-11-11T00:00:00"", ""end"": ""1999-07-19T0..."
6,changes,100.0,dict,"{""2012-03-09T18:01:40"": [{""type"": ""added"", ""data"": [{""Organization"": ""Transport and Tourism"", ""role"": ""Member"", ""abb..."
7,Groups,99.9,list,"[{""role"": ""Member"", ""Organization"": ""Group of the European People's Party (Christian-Democratic Group)"", ""groupid"": ..."
8,Birth,99.8,dict,"{""date"": ""1948-10-03T00:00:00"", ""place"": ""Gries""}"
9,Gender,99.8,str,"""M"""


## `ep_dossiers`

In [8]:
show_raw("ep_dossiers", n=1)

===== ep_dossiers: first 1 records =====
{
 "activities": [
  {
   "date": "2011-03-28T00:00:00",
   "docs": [
    {
     "text": [
      "<div id=\"summary\">\n<p style=\"text-align:justify\" class=\"MsoNormal\"><span lang=\"EN-GB\">PURPOSE: to submit to the EU legislature draft\namendments to the Statute of the Court of Justice and Annex I\nthereto.</span></p>\n<p style=\"text-align:justify\" class=\"MsoNormal\"><span lang=\"EN-GB\">PROPOSED ACT: Regulation of the European Parliament\nand of the Council.</span></p>\n<p style=\"text-align:justify\" class=\"MsoNormal\"><span lang=\"EN-GB\">LEGAL BASIS: Article 19(2), second paragraph, of the\nTreaty on the Functioning of the European Union. (TFUE); first\nparagraph of Article 254, the first and second paragraphs of\nArticle 257 and the second paragraph of Article 281 TFEU; Article\n106a(1) of the Treaty establishing the European Atomic Energy\nCommunity. &#160;</span></p>\n<p style=\"text-align:justify\" class=\"MsoNormal\"><span lang=

In [9]:
fields("ep_dossiers")

,field,present_%,type,example
0,meta,100.0,dict,"{""source"": ""http://www.europarl.europa.eu/oeil/popups/ficheprocedure.do?reference=2011/0901(COD)"", ""updated"": ""2012-..."
1,changes,100.0,dict,"{""2012-04-01T23:37:53"": [{""data"": ""http://www.europarl.europa.eu/sides/getDoc.do?type=COMPARL&mode=XML&language=EN&r..."
2,procedure,100.0,dict,"{""dossier_of_the_committee"": ""JURI/7/05776"", ""reference"": ""2011/0901(COD)"", ""subtype"": ""Legislation"", ""title"": ""Cour..."
3,events,93.8,list,"[{""date"": ""2014-01-08T00:00:00"", ""type"": ""Legislative proposal published"", ""body"": ""EC"", ""docs"": [{""url"": ""http://re..."
4,docs,80.0,list,"[{""date"": ""2014-01-08T00:00:00"", ""docs"": [{""url"": ""http://register.consilium.europa.eu/content/out?lang=EN&typ=SET&i..."
5,committees,73.8,list,"[{""body"": ""EP"", ""responsible"": false, ""committee"": ""AFCO"", ""date"": ""2011-07-12T00:00:00"", ""committee_full"": ""Constit..."
6,commission,53.1,list,"[{""body"": ""EC"", ""dg"": ""Economic and Financial Affairs"", ""commissioner"": ""REHN Olli""}]"
7,council,25.2,list,"[{""body"": ""CSL"", ""type"": ""Council Meeting"", ""council"": ""Economic and Financial Affairs ECOFIN"", ""meeting_id"": ""3324""..."
8,links,24.1,dict,{}
9,otherinst,8.7,list,"[{""name"": ""European Economic and Social Committee""}]"


## AI Act subset as a table
Amendments for one procedure, flattened into a DataFrame (all raw fields kept as columns).

In [10]:
REF = "2021/0106(COD)"
rows = [r for r in records("ep_amendments") if r.get("reference") == REF]
df = pd.json_normalize(rows)
print(len(df), "amendments")
print(list(df.columns))
df.head(10)

4852 amendments
['src', 'peid', 'reference', 'date', 'committee', 'seq', 'id', 'orig_lang', 'old', 'new', 'authors', 'meps', 'location', 'meta.created', 'compromise', 'justification']


,src,peid,reference,date,committee,seq,id,orig_lang,old,new,authors,meps,location,meta.created,compromise,justification
0,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf,PE704.585v01-00,2021/0106(COD),2022-01-25T00:00:00,[ENVI],68,PE704.585-68,en,"[(1) The purpose of this Regulation is to, improve the functioning of the internal, market by laying down a uniform ...","[(1) The purpose of this Regulation is to, improve the functioning of the internal, market by laying down a uniform ...","César Luena, Javi López, Cyrus Engerer, Milan Brglez","[197721, 125042, 209091, 197446]","[[Proposal for a regulation, Recital 1]]",2022-02-24T00:23:49,NaN,NaN
1,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf,PE704.585v01-00,2021/0106(COD),2022-01-25T00:00:00,[ENVI],69,PE704.585-69,en,"[(1) The purpose of this Regulation is to, improve the functioning of the internal, market by laying down a uniform ...","[(1) The purpose of this Regulation is to, improve the functioning of the internal, market by laying down a uniform ...",Margrete Auken on behalf of the Greens/EFA Group,[28161],"[[Proposal for a regulation, Recital 1]]",2022-02-24T00:23:49,NaN,NaN
2,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf,PE704.585v01-00,2021/0106(COD),2022-01-25T00:00:00,[ENVI],70,PE704.585-70,en,NaN,"[(1 a) This Regulation should foster a, supportive environment for healthier, lifestyles in a sustainable and climat...",Margrete Auken on behalf of the Greens/EFA Group,[28161],"[[ Proposal for a regulation, Recital 1 a (new)]]",2022-02-24T00:23:49,NaN,NaN
3,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf,PE704.585v01-00,2021/0106(COD),2022-01-25T00:00:00,[ENVI],71,PE704.585-71,en,"[(2) Artificial intelligence systems (AI, systems) can be easily deployed in multiple, sectors of the economy and so...","[(2) Artificial intelligence systems (AI, systems) can be easily deployed in multiple, sectors of the economy and so...","César Luena, Javi López, Cyrus Engerer, Milan Brglez","[197721, 125042, 209091, 197446]","[[ Proposal for a regulation, Recital 2]]",2022-02-24T00:23:49,NaN,NaN
4,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf,PE704.585v01-00,2021/0106(COD),2022-01-25T00:00:00,[ENVI],72,PE704.585-72,en,"[(3) Artificial intelligence is a fast, evolving family of technologies that can, contribute to a wide array of econ...","[(3) Artificial intelligence is a fast, evolving family of technologies that can, contribute to a wide array of econ...",Margrete Auken,[28161],"[[ Proposal for a regulation, Recital 3]]",2022-02-24T00:23:49,[ on behalf of the Greens/EFA Group],NaN
5,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf,PE704.585v01-00,2021/0106(COD),2022-01-25T00:00:00,[ENVI],73,PE704.585-73,en,"[(3) Artificial intelligence is a fast, evolving family of technologies that can, contribute to a wide array of econ...","[(3) Artificial intelligence is a fast, evolving family of technologies that can, contribute to a wide array of econ...",Michal Wiezik,[197769],"[[ Proposal for a regulation, Recital 3]]",2022-02-24T00:23:49,NaN,NaN
6,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf,PE704.585v01-00,2021/0106(COD),2022-01-25T00:00:00,[ENVI],74,PE704.585-74,en,"[(3) Artificial intelligence is a fast, evolving family of technologies that can, contribute to a wide array of econ...","[(3) Artificial intelligence is a fast, evolving family of technologies that can, contribute to a wide array of econ...","César Luena, Javi López, Cyrus Engerer, Milan Brglez","[197721, 125042, 209091, 197446]","[[Proposal for a regulation, Recital 3]]",2022-02-24T00:23:49,NaN,NaN
7,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf,PE704.585v01-00,2021/0106(COD),2022-01-25T00:00:00,[ENVI],75,PE704.585-75,en,NaN,"[(3 a) In order to ensure the dual, ecological and digital transition, secure, the technological resilience of the U...","Marie Toussaint

In [11]:
# one amendment in full, with the text lines joined
r = rows[0]
for k, v in r.items():
    print(f"{k:15} {' '.join(v) if k in ('old', 'new') and isinstance(v, list) else v}")

src             https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf
peid            PE704.585v01-00
reference       2021/0106(COD)
date            2022-01-25T00:00:00
committee       ['ENVI']
seq             68
id              PE704.585-68
orig_lang       en
old             (1) The purpose of this Regulation is to improve the functioning of the internal market by laying down a uniform legal framework in particular for the development, marketing and use of artificial intelligence in conformity with Union values. This Regulation pursues a number of overriding reasons of public interest, such as a high level of protection of health, safety and fundamental rights, and it ensures the free movement of AI- based goods and services cross-border, thus preventing Member States from imposing restrictions on the development, marketing and use of AI systems, unless explicitly authorised by this Regulation.
new             (1) The purpose of this Regulation is to improve the function

## Output files from `build_dashboard.py` (if already run)

In [12]:
for p in sorted(Path("output").glob("*.csv")):
    out = pd.read_csv(p)
    print(p, out.shape)
    print(list(out.columns))
    display(out.head(5))

output/amendments_2021_0106_COD.csv (4852, 17)
['id', 'pe', 'seq', 'date', 'committee', 'location', 'provision', 'is_new', 'deletion', 'authors', 'mep_names', 'groups', 'old', 'new', 'added_words', 'justification', 'src']


,id,pe,seq,date,committee,location,provision,is_new,deletion,authors,mep_names,groups,old,new,added_words,justification,src
0,PE704.585-68,PE704.585v01-00,68,2022-01-25,ENVI,Recital 1,Recital 1,False,False,"César Luena, Javi López, Cyrus Engerer, Milan Brglez",César LUENA; Javi LÓPEZ; Cyrus ENGERER; Milan BRGLEZ,S&D; S&D; S&D; S&D,(1) The purpose of this Regulation is to improve the functioning of the internal market by laying down a uniform leg...,(1) The purpose of this Regulation is to improve the functioning of the internal market by laying down a uniform leg...,"design, … and of sustainable and green artificial intelligence … values while minimising any risk of adverse and dis...",NaN,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf
1,PE704.585-69,PE704.585v01-00,69,2022-01-25,ENVI,Recital 1,Recital 1,False,False,Margrete Auken on behalf of the Greens/EFA Group,Margrete AUKEN,Greens/EFA,(1) The purpose of this Regulation is to improve the functioning of the internal market by laying down a uniform leg...,(1) The purpose of this Regulation is to improve the functioning of the internal market by laying down a uniform leg...,"priorities and … biodiversity, the climate and the environment,",NaN,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf
2,PE704.585-70,PE704.585v01-00,70,2022-01-25,ENVI,Recital 1 a (new),Recital 1 a (new),True,False,Margrete Auken on behalf of the Greens/EFA Group,Margrete AUKEN,Greens/EFA,NaN,(1 a) This Regulation should foster a supportive environment for healthier lifestyles in a sustainable and climate n...,(1 a) This Regulation should foster a supportive environment for healthier lifestyles in a sustainable and climate n...,NaN,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf
3,PE704.585-71,PE704.585v01-00,71,2022-01-25,ENVI,Recital 2,Recital 2,False,False,"César Luena, Javi López, Cyrus Engerer, Milan Brglez",César LUENA; Javi LÓPEZ; Cyrus ENGERER; Milan BRGLEZ,S&D; S&D; S&D; S&D,(2) Artificial intelligence systems (AI systems) can be easily deployed in multiple sectors of the economy and socie...,(2) Artificial intelligence systems (AI systems) can be easily deployed in multiple sectors of the economy and socie...,"persons, end users and end recipients",NaN,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf
4,PE704.585-72,PE704.585v01-00,72,2022-01-25,ENVI,Recital 3,Recital 3,False,False,Margrete Auken,Margrete AUKEN,Greens/EFA,(3) Artificial intelligence is a fast evolving family of technologies that can contribute to a wide array of economi...,(3) Artificial intelligence is a fast evolving family of technologies that can contribute to a wide array of economi...,"food safety, by reducing the use of pesticides, the conservation and restoration of biodiversity and ecosystems, acc...",NaN,https://www.europarl.europa.eu/doceo/document/ENVI-AM-704585_EN.pdf


## Have Your Say feedback (if `fetch_feedback.py` was run)

In [13]:
for p in sorted(Path("feedback").glob("*/feedback.csv")):
    fb = pd.read_csv(p)
    print(p, fb.shape)
    print(list(fb.columns))
    display(fb.head(5))